In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_table_name = dbutils.widgets.get("source_table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=== Input Parameters ===")
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  source_table_name : {source_table_name}")
print(f"  merge_keys        : {merge_keys}")
print(f"  load_type         : {load_type}")

In [0]:
bronze_table = f"caresync_adb.bronze.{source_table_name}"
print(f"=== Reading Bronze Table: {bronze_table} ===")
bronze_df = spark.read.table(bronze_table)
print(f"  Row count : {bronze_df.count()}")
print(f"  Columns   : {bronze_df.columns}")

In [0]:
from pyspark.sql.functions import *

target_table = f"caresync_adb.silver.{source_table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_keys_list = [key.strip() for key in merge_keys.split(',') if key.strip()]

print("=== Transformation Config ===")
print(f"  Target table         : {target_table}")
print(f"  Normalized load type : {normalized_load_type}")
print(f"  Merge keys list      : {merge_keys_list}")

if normalized_load_type == "MERGE":
    if not merge_keys_list:
        raise ValueError("merge_keys must be provided when the load_type is 'MERGE'")

    missing_merge_keys = [key for key in merge_keys_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("updated_timestamp", current_timestamp())

In [0]:
print(f"=== Executing Load Type: {normalized_load_type} ===")

if normalized_load_type == 'FULL':
    print(f"  Overwriting target table: {target_table}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed successfully.")

elif normalized_load_type == "APPEND":
    print(f"  Appending to target table: {target_table}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed successfully.")

elif normalized_load_type == "MERGE":
    table_exists = spark.catalog.tableExists(target_table)
    print(f"  Target table exists: {table_exists}")

    if not table_exists:
        print(f"  Target table not found. Creating with initial overwrite.")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  MERGE (initial load) completed successfully.")

    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)

    merge_condition = " AND ".join(
                                    [f"target.`{key}` = source.`{key}`" for key in merge_keys_list]
    )   

    update_assignments = ", ".join(
                                    [f"target.`{column}` = source.`{column}`" 
                                        for column in updated_df.columns
                                        if column != "insert_timestamp"]
    )

    insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
    insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

    print(f"  Merge condition : {merge_condition}")
    print(f"  Insert columns  : {insert_columns}")

    merge_result = spark.sql(f"""
              MERGE INTO {target_table} AS target
              USING {source_view} AS source
              ON {merge_condition}
              WHEN MATCHED THEN UPDATE SET {update_assignments}
              WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
              """
              )
    print("  MERGE completed successfully.")
    display(merge_result)

In [0]:
record_count = updated_df.count()
print(f"=== Notebook Exit ===")
print(f"  Total records processed: {record_count}")
dbutils.notebook.exit(record_count)